# RAG em Bancos de Dados Multimodais: Orquestrando Recuperação Textual, Vetorial e em Grafo

### Dataset e Tecnologias

Para este projeto, usaremos um dataset de recomendação de filmes. Os dados dos filmes (título, gêneros) serão armazenados no PostgreSQL, enquanto as relações (usuário avaliou filme) serão modeladas como um grafo no Neo4j. As tecnologias que utilizaremos incluem:

- **Supabase**: Como nosso provedor de PostgreSQL com a extensão `pg_vector` já habilitada.

- **Neo4j AuraDB**: Uma base de dados em grafo como serviço.

- **API da OpenAI**: Utilizada para gerar os embeddings vetoriais dos títulos dos filmes e para a etapa final de geração de texto com um LLM.

- **LangChain**: O framework que usaremos para orquestrar nosso pipeline de RAG, conectando as múltiplas fontes de dados ao LLM.

## Seção 1: Preparação do Ambiente

Nesta seção, vamos instalar todas as bibliotecas Python necessárias e configurar as credenciais para nos conectarmos aos nossos bancos de dados externos: Supabase (PostgreSQL) e Neo4j AuraDB, além da API da OpenAI.

### Instalação das Bibliotecas
Primeiro, vamos instalar os pacotes Python que usaremos ao longo do notebook.

In [3]:
# Instalação das bibliotecas necessárias
!pip install -q neo4j openai langchain langchain-openai psycopg2-binary pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 325.4/325.4 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.7/84.7 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 62.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 477.4/477.4 kB 25.9 MB/s eta 0:00:00


### Configuração das Chaves de Acesso (Secrets)
Para nos conectarmos aos serviços, precisaremos de chaves de API e credenciais. A maneira mais segura de gerenciá-las no Google Colab é usando o gerenciador de "Secrets" (no menu à esquerda, ícone de chave 🔑).

Por favor, adicione as seguintes chaves no gerenciador de Secrets:

#### OpenAI:

`OPENAI_API_KEY`: Sua chave da API da OpenAI.

#### Supabase (PostgreSQL):

`SUPABASE_HOST`: O host do seu banco de dados.

`SUPABASE_DATABASE`: O nome do banco de dados (geralmente 'postgres').

`SUPABASE_USER`: O usuário do banco de dados (geralmente 'postgres').

`SUPABASE_PASSWORD`: A senha do seu projeto Supabase.

#### Neo4j AuraDB:

`NEO4J_URI`: A URI de conexão do seu banco de dados AuraDB.

`NEO4J_USERNAME`: O usuário do banco de dados (geralmente 'neo4j').

`NEO4J_PASSWORD`: A senha que você gerou para a instância.

Agora, vamos carregar essas chaves em nosso ambiente:

In [4]:
from google.colab import userdata
import os

try:
    # Carregando OpenAI
    os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')

    # Carregando Supabase (Postgres)
    SUPABASE_USER = userdata.get('SUPABASE_USER')
    SUPABASE_PASSWORD = userdata.get('SUPABASE_PASSWORD')
    SUPABASE_HOST = userdata.get('SUPABASE_HOST')
    SUPABASE_DATABASE = userdata.get('SUPABASE_DATABASE')

    # Carregando Neo4j
    NEO4J_URI = userdata.get('NEO4J_URI')
    NEO4J_USERNAME = userdata.get('NEO4J_USERNAME')
    NEO4J_PASSWORD = userdata.get('NEO4J_PASSWORD')

    print("✅ Todas as credenciais foram carregadas com sucesso!")

except Exception as e:
    print(f"❌ Erro ao carregar segredos: {e}")
    print("Verifique se os nomes no menu 'Secrets' correspondem exatamente aos nomes acima.")

✅ Todas as credenciais foram carregadas com sucesso!


In [6]:
import pandas as pd
from openai import OpenAI

# Inicializa o cliente usando a chave carregada no ambiente
client = OpenAI()

def get_embeddings_batch(texts, model="text-embedding-3-small"):
    cleaned = []
    for t in texts:
        # Garante que é string e remove quebras de linha
        if pd.isna(t) or t == "":
            cleaned.append(" ") # OpenAI não aceita string vazia
        else:
            cleaned.append(str(t).replace("\n", " "))

    try:
        resp = client.embeddings.create(model=model, input=cleaned)
        return [item.embedding for item in resp.data]
    except Exception as e:
        print(f"Erro na API da OpenAI: {e}")
        return []

# Carrega o CSV
try:
    df = pd.read_csv("Books Dataset for NLP & Recommendation Systems.csv")
    df_subset = df.head(100).copy()

    print("Gerando embeddings... (isso pode custar créditos)")
    texts = df_subset["description"].fillna(" ").tolist()

    # Gera embeddings em lotes ou de uma vez (cuidado com limites)
    df_subset["embedding"] = get_embeddings_batch(texts)

    if len(df_subset["embedding"]) > 0:
        print("✅ Embeddings gerados com sucesso!")
    else:
        print("⚠️ Falha ao gerar embeddings.")

except FileNotFoundError:
    print("❌ Erro: O arquivo CSV não foi encontrado. Faça o upload no menu lateral.")

Gerando embeddings... (isso pode custar créditos)
Erro na API da OpenAI: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}


ValueError: Length of values (0) does not match length of index (100)